In [ ]:
ecli, link, creator, date, issued, subject, procedure, type, inhoudsindicatie, uitspraak

In [2]:
from app.services.rechtspraak_fetcher import fetch_cases, FEED_FIELDS, DOC_FIELDS, ALL_FIELDS

# Fast - only feed fields (no extra requests)
#cases = fetch_cases(fields=FEED_FIELDS, max_results=100)

# All fields (slower - fetches each document)
#cases = fetch_cases(fields=ALL_FIELDS, max_results=100)

# Specific fields
cases = fetch_cases(
    fields=ALL_FIELDS,
    max_results=100
)

# Convert to JSON
import json
print(json.dumps(cases, indent=2, ensure_ascii=False))


[
  {
    "ecli": "ECLI:NL:RBSHE:2012:7589",
    "link": "https://uitspraken.rechtspraak.nl/details?id=ECLI:NL:RBSHE:2012:7589",
    "creator": "Rechtbank 's-Hertogenbosch",
    "date": "2012-06-19",
    "issued": "2013-07-26",
    "subject": "Civiel recht; Arbeidsrecht",
    "procedure": "Eerste aanleg - enkelvoudig",
    "type": "Uitspraak",
    "inhoudsindicatie": "Ontbindingsverzoek werknemer op 7 mei 2012, nadat de werkgever op 12 april 2012 het UWV heeft verzocht om toestemming om de arbeidsovereenkomst op te zeggen. Op 14 mei 2012 heeft de werkgever toestemming gekregen. Zij heeft de arbeidsovereenkomst op 16 mei 2012 opgezegd per 17 mei 2012. De mondelinge behandeling van het ontbindingsverzoek heeft plaatsgevonden op 31 mei 2012. De kantonrechter is van oordeel dat werknemer ontvankelijk is, omdat de arbeidsovereenkomst nog bestond toen hij het verzoek indiende. Dat de arbeidsovereenkomst inmiddels door onregelmatige opzegging is geëindigd doet daaraan niet af. De opzegging wa

In [17]:
import re
from typing import Iterable, Set, Tuple, Dict

_WORD_RE = re.compile(r"[a-zA-ZÀ-ÿ]+", re.UNICODE)

def normalize_tokens(text: str) -> list[str]:
    text = text.lower()
    return _WORD_RE.findall(text)

def light_nl_stem(token: str) -> str:
    """
    Super-lichte NL 'stemmer': goed genoeg voor keyword-gates.
    Let op: het is heuristisch, niet taalkundig perfect.
    """
    t = token

    # Veelvoorkomende suffixen in uitspraken / contracttaal
    for suf in ("heden", "ing", "ingen", "lijkheid", "lijk", "heden", "eren", "eren", "baar", "baarheden"):
        if len(t) > 6 and t.endswith(suf):
            t = t[: -len(suf)]
            break

    # Meervoud/verbogen vormen
    for suf in ("en", "s", "e", "n"):
        if len(t) > 5 and t.endswith(suf):
            t = t[: -len(suf)]
            break

    # Diminishing returns: niet te kort maken
    return t if len(t) >= 3 else token

def build_stem_set(words: Iterable[str]) -> Set[str]:
    return {light_nl_stem(w.lower()) for w in words}

def keyword_gate(
    text: str,
    *,
    anchor_words: Iterable[str],
    theme_words: Iterable[str],
    strong_words: Iterable[str] = (),
    min_theme_hits: int = 2,
    min_score: float = 2.5,
) -> Tuple[bool, Dict[str, object]]:
    """
    Match op basis van "stem-voting":
    - anchors: arbeidscontract-context (arbeidsovereenkomst, beding, werkgever, etc.)
    - theme: onduidelijkheid/fouten/omissie/uitleg
    - strong: zwaardere signalen (bv. haviltex, contra proferentem, kennelijke fout)
    """

    tokens = normalize_tokens(text)
    stems = [light_nl_stem(t) for t in tokens]
    stemset = set(stems)

    anchors = build_stem_set(anchor_words)
    themes = build_stem_set(theme_words)
    strongs = build_stem_set(strong_words)

    anchor_hits = stemset.intersection(anchors)
    theme_hits = stemset.intersection(themes)
    strong_hits = stemset.intersection(strongs)

    # Score: uniek hits tellen (minder gevoelig voor lange teksten)
    score = 1.0 * len(theme_hits) + 1.5 * len(strong_hits) + 0.5 * len(anchor_hits)

    # Extra: relatieve dekking helpt een beetje (maar minimalistisch)
    # (voorkomt dat 2 hits in 5000 woorden teveel gewicht krijgen)
    coverage = (len(theme_hits) + len(strong_hits) + len(anchor_hits)) / max(len(stemset), 1)
    score += 10.0 * coverage  # kleine boost; pas aan indien nodig

    is_match = (
        len(anchor_hits) >= 1 and
        len(theme_hits) >= min_theme_hits and
        score >= min_score
    )

    debug = {
        "score": round(score, 3),
        "anchor_hits": sorted(anchor_hits),
        "theme_hits": sorted(theme_hits),
        "strong_hits": sorted(strong_hits),
        "unique_tokens": len(stemset),
        "coverage": round(coverage, 4),
    }
    return is_match, debug


if __name__ == "__main__":
    anchor_words = [
        "arbeidsovereenkomst", "arbeidscontract", "beding", "werkgever", "werknemer",
        "salaris", "loon", "proeftijd", "concurrentiebeding", "relatiebeding", "cao"
    ]

    theme_words = [
        "onduidelijk", "dubbelzinnig", "meerduidig", "uitleg", "interpretatie",
        "omissie", "ontbreekt", "niet vermeld", "tegenstrijdig", "verschrijving",
        "typefout", "rekenfout", "misslag", "onjuist", "verzuim"
    ]

    strong_words = [
        "haviltex", "contra proferentem", "kennelijke fout", "redelijkheid", "billijkheid",
        "dwaling", "wilsgebrek"
    ]

    sample = """
    Partijen verschillen van mening over de uitleg van het concurrentiebeding in de arbeidsovereenkomst.
    Het beding is op onderdelen onduidelijk en bevat een kennelijke verschrijving.
    """

    m, info = keyword_gate(
        sample,
        anchor_words=anchor_words,
        theme_words=theme_words,
        strong_words=strong_words,
        min_theme_hits=0,
        min_score=0,
    )
    print(m, info)


True {'score': 7.357, 'anchor_hits': ['arbeidsovereenkomst', 'beding', 'concurrentiebed'], 'theme_hits': ['onduid', 'uitleg', 'verschrijv'], 'strong_hits': [], 'unique_tokens': 21, 'coverage': 0.2857}


In [ ]:
keywords = [
    # kernstructuur / contractreferenties
    "artikel", "artikelen", "lid", "leden", "paragraaf", "clausule", "beding",
    "bijlage", "bijlagen", "addendum", "aanhangsel", "schema", "regeling",
    "definitie", "definities", "begrip", "begrippen", "interpretatie",

    # functie & scope
    "functie", "functietitel", "functieomschrijving", "taken", "werkzaamheden",
    "standplaats", "werklocatie", "hybride", "thuiswerken", "overplaatsing",
    "inzetbaarheid", "nevenwerkzaamheden",

    # arbeidsduur & werktijden
    "arbeidsduur", "uren", "fulltime", "parttime", "rooster", "werktijden",
    "overwerk", "meerwerk", "oproep", "bereikbaarheid", "consignatie",

    # duur / aanvang / einde
    "ingangsdatum", "aanvang", "looptijd", "bepaalde tijd", "onbepaalde tijd",
    "verlenging", "proeftijd", "beëindiging", "ontbinding", "opzegging",
    "opzegtermijn", "wederzijds goedvinden",

    # beloning & vergoedingen
    "salaris", "loon", "bruto", "netto", "schaal", "periodiek", "bonus",
    "variabele beloning", "commissie", "13e maand", "vakantiegeld",
    "onkosten", "reiskosten", "thuiswerkvergoeding", "overwerktoeslag",
    "indexatie",

    # verlof & afwezigheid
    "vakantiedagen", "verlof", "bovenwettelijk", "wettelijk", "feestdagen",
    "ziekte", "arbeidsongeschiktheid", "loonbetaling bij ziekte",
    "re-integratie", "zwangerschapsverlof", "ouderschapsverlof",

    # pensioen & verzekeringen
    "pensioen", "pensioenregeling", "premie", "verzekeringen",
    "arbeidsongeschiktheidsverzekering",

    # gedrag / compliance
    "gedragscode", "integriteit", "compliance", "klokkenluidersregeling",
    "nevenfuncties", "screening", "vbg", "achtergrondonderzoek",

    # vertrouwelijkheid, data, ip
    "geheimhouding", "vertrouwelijkheid", "confidentialiteit",
    "privacy", "avg", "persoonsgegevens", "gegevensverwerking",
    "intellectuele eigendom", "auteursrecht", "octrooi", "ip",
    "broncode", "uitvindingen",

    # restricties (vaak ‘te streng’ of juist ontbrekend)
    "concurrentiebeding", "relatiebeding", "non-compete", "non-solicit",
    "boetebeding", "boete", "dwangsom", "schadevergoeding",
    "terugbetaling", "studiekostenbeding", "opleidingskosten",
    "inhouding", "verrekening",

    # prestaties & beoordeling
    "prognose", "targets", "kpi", "beoordeling", "functioneringsgesprek",
    "verbetertraject", "p ip", "plan van aanpak", "disfunctioneren",

    # wijziging / eenzijdigheid (vaak rode vlag)
    "wijzigingsbeding", "eenzijdig", "naar eigen inzicht", "te allen tijde",
    "zonder opgaaf van reden", "kan", "mag", "zal", "moet", "uitsluitend",

    # aansprakelijkheid & risico
    "aansprakelijkheid", "beperking van aansprakelijkheid",
    "vrijwaring", "indemniteit", "risico", "zorgplicht",

    # cao / wet / verwijzingen
    "cao", "handboek", "personeelshandboek", "reglement",
    "arbeidstijdenwet", "bw", "burgerlijk wetboek", "wet", "wettelijk",
    "dwingend recht",

    # uitzonderingen / vaagheid / kwaliteitssignalen
    "tenzij", "voor zover", "behoudens", "onder meer", "zoals", "bijvoorbeeld",
    "in beginsel", "naar redelijkheid", "naar billijkheid",
    "passend", "adequaat", "tijdig", "zo spoedig mogelijk",
    "redelijke termijn", "eventueel", "naar behoefte",

    # rangorde / conflict tussen documenten
    "rangorde", "prevaleert", "voorrang", "tegenstrijdigheid",
    "in geval van strijd", "interpretatieverschil",

    "arbeidsovereenkomst", "arbeidscontract", "beding", "werkgever", "werknemer",
    "salaris", "loon", "proeftijd", "concurrentiebeding", "relatiebeding", "cao"

    "onduidelijk", "dubbelzinnig", "meerduidig", "uitleg", "interpretatie",
    "omissie", "ontbreekt", "niet vermeld", "tegenstrijdig", "verschrijving",
    "typefout", "rekenfout", "misslag", "onjuist", "verzuim"

    "haviltex", "contra proferentem", "kennelijke fout", "redelijkheid", "billijkheid",
    "dwaling", "wilsgebrek"

]


In [10]:
for c in cases:

    sample = c.get("uitspraak", "")

    m, info = keyword_gate(
    sample,
    anchor_words=anchor_words,
    theme_words=theme_words,
    strong_words=strong_words,
    min_theme_hits=2,
    min_score=2.5,
    )
    
    if m:
        print("=== MATCH ===")
        print(c['inhoudsindicatie'])


=== MATCH ===
Ontslag wegens bedrijfseconomische omstandigheden. Uitleg Cao. Overgangsregeling. umulatie wachtgeld met ontslagvergoeding. Toepassing Cao-maatstaf.
=== MATCH ===
arbeidsrecht, vergoeding van kosten van rechtsbijstand, onvoorwaardelijke toezegging om kosten van rechtsbijstand te vergoeden
=== MATCH ===
Min/max-contract in de thuiszorg. Verrekening van min-uren met vakantie-uren is onder de omstandigheden van dit geval niet toelaatbaar.
=== MATCH ===
Bestuurders aansprakelijkheid; uitgavenpetroon; niet als goed bestuurder gehandeld. Welke posten komen voor terugbetaling in aanmerking.
=== MATCH ===
Uitleg van een non-concurrentie bepaling ivm een franchise overeenkomst. Haviltex-criterium.
=== MATCH ===
Au pair blijft na au-pairperiode werkzaam gedurende enkele jaren. Vordering minimumloon toewijsbaar (arbeidsovereenkomst). Vakantiedagen. Geen retentierecht werkneemster mbt auto.
=== MATCH ===
ontslag op staande voet - druppel die de emmer doet overlopen
=== MATCH ===
EJ. 

In [1]:
from app.services.rechtspraak_fetcher import fetch_document, get_paragraphs, get_sections

# Fetch the raw XML
doc = fetch_document("ECLI:NL:RBSHE:2012:7589")

# Get individual paragraphs (smallest chunks)
paragraphs = get_paragraphs(doc)  # list[str] - 45 paragraphs in this case

# Get sections (larger logical blocks)
sections = get_sections(doc)  # list[str] - 16 sections in this case


In [20]:
for s in paragraphs:
    print("=== SECTION ===")
    print(s)

=== SECTION ===
Ontbindingsverzoek werknemer op 7 mei 2012, nadat de werkgever op 12 april 2012 het UWV heeft verzocht om toestemming om de arbeidsovereenkomst op te zeggen. Op 14 mei 2012 heeft de werkgever toestemming gekregen. Zij heeft de arbeidsovereenkomst op 16 mei 2012 opgezegd per 17 mei 2012. De mondelinge behandeling van het ontbindingsverzoek heeft plaatsgevonden op 31 mei 2012. De kantonrechter is van oordeel dat werknemer ontvankelijk is, omdat de arbeidsovereenkomst nog bestond toen hij het verzoek indiende. Dat de arbeidsovereenkomst inmiddels door onregelmatige opzegging is geëindigd doet daaraan niet af. De opzegging was weliswaar onrechtmatig, maar desalniettemin geldig. Werknemer heeft geen gronden aangevoerd waarom de opzegging niet geldig zou zijn. Gelet daarop bestond de arbeidsovereenkomst ten tijde van de mondelinge behandeling niet meer en kan de kantonrechter niet tot ontbinding daarvan overgaan.
=== SECTION ===
Sector Kanton, locatie ‘s-Hertogenbosch
=== SEC

In [23]:
from app.services.rechtspraak_fetcher import fetch_document, DOC_NS
from lxml import etree

# Fetch a sample document
doc = fetch_document("ECLI:NL:HR:2023:1")  # or use an ECLI from your cases

# See the root tag and namespace
print(f"Root tag: {doc.tag}")
print(f"Root attribs: {doc.attrib}")

# List all unique child element tags
def list_all_tags(elem, prefix=""):
    tags = set()
    for child in elem:
        tag = child.tag
        tags.add(tag)
        tags.update(list_all_tags(child, prefix + "  "))
    return tags

print("\nAll unique tags in document:")
for tag in sorted(list_all_tags(doc)):
    print(f"  {tag}")

# Show top-level structure
print("\nTop-level children:")
for child in doc:
    print(f"  {child.tag}")


Root tag: open-rechtspraak
Root attribs: {}

All unique tags in document:
  {http://psi.rechtspraak.nl/}procedure
  {http://psi.rechtspraak.nl/}zaaknummer
  {http://purl.org/dc/terms/}accessRights
  {http://purl.org/dc/terms/}coverage
  {http://purl.org/dc/terms/}creator
  {http://purl.org/dc/terms/}date
  {http://purl.org/dc/terms/}format
  {http://purl.org/dc/terms/}identifier
  {http://purl.org/dc/terms/}issued
  {http://purl.org/dc/terms/}language
  {http://purl.org/dc/terms/}modified
  {http://purl.org/dc/terms/}publisher
  {http://purl.org/dc/terms/}subject
  {http://purl.org/dc/terms/}type
  {http://www.w3.org/1999/02/22-rdf-syntax-ns#}Description
  {http://www.w3.org/1999/02/22-rdf-syntax-ns#}RDF

Top-level children:
  {http://www.w3.org/1999/02/22-rdf-syntax-ns#}RDF


In [24]:
# Pretty print the whole XML (first 5000 chars)
xml_str = etree.tostring(doc, pretty_print=True, encoding="unicode")
print(xml_str[:5000])


<open-rechtspraak>
  <rdf:RDF xmlns:rdf="http://www.w3.org/1999/02/22-rdf-syntax-ns#" xmlns:ecli="https://e-justice.europa.eu/ecli" xmlns:tr="http://tuchtrecht.overheid.nl/" xmlns:eu="http://publications.europa.eu/celex/" xmlns:dcterms="http://purl.org/dc/terms/" xmlns:bwb="bwb-dl" xmlns:cvdr="http://decentrale.regelgeving.overheid.nl/cvdr/" xmlns:psi="http://psi.rechtspraak.nl/" xmlns:rdfs="http://www.w3.org/2000/01/rdf-schema#">
    <rdf:Description>
      <dcterms:identifier>ECLI:NL:HR:2023:1</dcterms:identifier>
      <dcterms:format>text/xml</dcterms:format>
      <dcterms:accessRights>public</dcterms:accessRights>
      <dcterms:modified>2023-03-04T00:00:28</dcterms:modified>
      <dcterms:issued rdfs:label="Publicatiedatum">2022-10-20</dcterms:issued>
      <dcterms:publisher resourceIdentifier="http://rechtspraak.nl/">Raad voor de Rechtspraak</dcterms:publisher>
      <dcterms:language>nl</dcterms:language>
      <dcterms:creator rdfs:label="Instantie" resourceIdentifier="http

In [28]:
from app.services.rechtspraak_fetcher import fetch_document, DOC_NS
from lxml import etree

# Fetch a sample document (use one from your cases)
doc = fetch_document(cases[0]["ecli"])

# Find the uitspraak element
uitspraak = doc.find(".//rs:uitspraak", DOC_NS)

if uitspraak is None:
    print("No uitspraak found in this document")
else:
    # Show structure: count of each element type within uitspraak
    def count_elements(elem):
        counts = {}
        for child in elem.iter():
            # Skip comments and processing instructions (their .tag is callable)
            if not isinstance(child.tag, str):
                continue
            tag = child.tag.split("}")[-1] if "}" in child.tag else child.tag
            counts[tag] = counts.get(tag, 0) + 1
        return counts

    print("Element counts in uitspraak:")
    for tag, count in sorted(count_elements(uitspraak).items()):
        print(f"  {tag}: {count}")

    # Show the XML structure (first 4000 chars)
    print("\n--- Raw XML structure ---")
    print(etree.tostring(uitspraak, pretty_print=True, encoding="unicode")[:4000])

Element counts in uitspraak:
  bridgehead: 1
  emphasis: 2
  nr: 18
  para: 77
  parablock: 16
  paragroup: 14
  section: 4
  title: 4
  uitspraak: 1
  uitspraak.info: 1

--- Raw XML structure ---
<uitspraak xmlns="http://www.rechtspraak.nl/schema/rechtspraak-1.0" xmlns:xsi="http://www.w3.org/2001/XMLSchema-instance" xmlns:xsd="http://www.w3.org/2001/XMLSchema" xmlns:xlink="http://www.w3.org/1999/xlink" id="ECLI:NL:RBSHE:2012:7589:DOC" lang="nl" xml:space="preserve">
  <uitspraak.info>
    <bridgehead role="bold">RECHTBANK ‘s-HERTOGENBOSCH</bridgehead>
    <para>Sector Kanton, locatie ‘s-Hertogenbosch</para>
    <para/>
    <parablock>
      <para>Zaaknummer	: 827195</para>
      <para>EJ verz. 	: 12-2021</para>
      <para>Uitspraak 	: 19 juni 2012</para>
    </parablock>
    <para/>
    <para/>
    <parablock>
      <para>in de zaak van:</para>
    </parablock>
    <para/>
    <parablock>
      <para>
        <emphasis role="bold">
          [verzoekster]
        </emphasis>
      </

In [15]:
for c in cases:
    ecli = c.get('ecli')
    doc = fetch_document(ecli)
    sections = get_sections(doc)
    print(max([len(s) for s in sections]))

1186
1137
1692
3053
259
1672
3733
2605
1857
3225


HTTPError: 403 Client Error: Forbidden for url: https://data.rechtspraak.nl/uitspraken/content?id=ECLI%3ANL%3AGHSHE%3A2013%3A5908

In [14]:
from app.services.rechtspraak_fetcher import fetch_document, parse_uitspraak

doc = fetch_document(cases[0]["ecli"])
uitspraak = parse_uitspraak(doc)

# Explore the structure
for section in uitspraak.sections:
    print(f"\n=== Section {section.nr}: {section.title} ===")
    for para in section.paragraphs:
        print(f"  [{para.nr}] {para.text[:100]}...")



=== Section 1: De procedure ===
  [None] Bij verzoekschrift, ingekomen ter griffie van de rechtbank, sector kanton, locatie 's-Hertogenbosch,...
  [None] Zijdens verweerster is een verweerschrift ingediend....
  [None] De mondelinge behandeling heeft plaatsgevonden op 31 mei 2012 , bij welke gelegenheid partijen de za...
  [None] Na gevoerd debat is de beschikking bepaald op heden....

=== Section 2: Inleiding ===
  [2.1.] Tussen partijen bestaat een arbeidsovereenkomst. Verzoekster is sedert 1 mei 2003  in dienst van ver...
  [2.2.] Verzoekster grondt het verzoek primair onvoorwaardelijk op de stelling dat er gewichtige redenen zij...
  [2.4.] Verweerster heeft verweer gevoerd....

=== Section 3: De beoordeling ===
  [3.1.1] Primair is verweerster van mening dat verzoekster niet-ontvankelijk is in haar verzoek aangezien er ...
  [3.1.2.] Verzoekster heeft hiertegen als verweer het volgende ingebracht. Primair heeft zij gesteld dat zij o...
  [3.1.3.] De kantonrechter is van oordeel d

In [16]:
s

"Aldus gegeven en in het openbaar uitgesproken op 19 juni 2012  door mr. G.J.M. van Meel, kantonrechter te 's-Hertogenbosch."

FILTER_WORDS = [
    
]

In [ ]:
FILTER_WORDS = [
    "arbeidscontract",
    "arbeidsovereenkomst",
    "dienstverband",
    "contract van arbeid",
    "werkovereenkomst",
    "looptijd",
    "proeftijd",
    "opzegtermijn",
    "salaris",
    "arbeidsvoorwaarden",
    "functieomschrijving",
    "werkuren",
    "cao",
    "concurrentiebeding",
    "vast contract",
    "ontslag",
    "beëindiging",
    "tijdelijk contract",
    "onbepaalde tijd",
    "arbeidsduur",
    "vakantiedagen",
    "ziekteverzuim",
    "loonbetaling",
    "beding",
    "rechtspositie"
]


In [13]:
test = [
    'a',
    'q',
    'ab',
    'abc',
    'abcd',
    'abcde',
    'q',
    'qr',
    'qrs',
    'qrst',
]
def ensure_min_length(lst, min_length, max_length=None):
    result = []
    i = 0
    while i < len(lst):
        current = lst[i]
        # Try to merge until we reach min_length
        while len(current) < min_length and i + 1 < len(lst):
            next_item = lst[i + 1]
            merged = current + next_item
            if max_length and len(merged) > max_length:
                break  # Can't merge without exceeding max
            current = merged
            i += 1
        # Only keep if within bounds
        if len(current) >= min_length and (max_length is None or len(current) <= max_length):
            result.append(current)
        i += 1
    return result

final_list = ensure_min_length(test, 500, 5000)
print(final_list)

['aqab', 'abc', 'abcd', 'qqr', 'qrs', 'qrst']


In [20]:
par_texts = []
for par in uitspraak.sections[2].paragraphs:
    par_texts.append(par.text)

final_pars = ensure_min_length(par_texts, 500, 5000)
for p in final_pars:
    print("=== PAR ===")
    print(len(p))
    print(p)

=== PAR ===
623
Primair is verweerster van mening dat verzoekster niet-ontvankelijk is in haar verzoek aangezien er geen arbeidsovereenkomst meer bestaat tussen partijen. Verweerster heeft de arbeidsovereenkomst opgezegd per 17 mei 2012. Weliswaar is sprake van een onregelmatige opzegging, waardoor verweerster schadeplichtig is jegens verzoekster, echter er is geen sprake van een vernietigbare opzegging, aangezien het UWV WERKbedrijf op 14 mei 2012 toestemming voor ontslag heeft verleend. Verzoekster heeft bovendien alle schade van verweerster voor het niet in acht nemen van de opzegtermijn direct financieel aan haar gecompenseerd.
=== PAR ===
727
Verzoekster heeft hiertegen als verweer het volgende ingebracht. Primair heeft zij gesteld dat zij ontvankelijk is in haar verzoek omdat de arbeidsovereenkomst in ieder geval bestond op het moment dat zij haar verzoek indiende. Dat verweerster verzoekster de pas heeft willen afsnijden door willens en wetens zonder in achtneming van de opzegte